# SpaceX Falcon 9 Data Collection — API
This notebook retrieves historical launch records from the public SpaceX API. It keeps collection reproducible and writes a raw JSON file plus a compact CSV. Network/API availability can vary.

In [ ]:
import json, requests, pandas as pd
from pathlib import Path
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
url = 'https://api.spacexdata.com/v4/launches'
response = requests.get(url, timeout=30)
response.raise_for_status()
launches = response.json()
print('Launch records retrieved:', len(launches))

In [ ]:
with open(DATA_DIR / 'spacex_launches_raw.json', 'w', encoding='utf-8') as f:
    json.dump(launches, f, indent=2)
launches_df = pd.json_normalize(launches)
launches_df.to_csv(DATA_DIR / 'spacex_launches_api_raw.csv', index=False)
launches_df.head()

## Falcon 9 subset and basic fields

In [ ]:
falcon9 = [x for x in launches if x.get('rocket')]
# Resolve rocket IDs using the rockets endpoint, then keep launches associated with Falcon 9.
rockets_response = requests.get('https://api.spacexdata.com/v4/rockets', timeout=30)
rockets_response.raise_for_status()
rockets = rockets_response.json()
falcon9_ids = {r['id'] for r in rockets if r.get('name') == 'Falcon 9'}
falcon9_launches = [x for x in launches if x.get('rocket') in falcon9_ids]
summary = pd.DataFrame([{'flight_number': x.get('flight_number'), 'name': x.get('name'), 'date_utc': x.get('date_utc'), 'success': x.get('success'), 'launchpad_id': x.get('launchpad')} for x in falcon9_launches])
summary.to_csv(DATA_DIR / 'falcon9_api_summary.csv', index=False)
print(summary.shape)
summary.head()

## Limitations
The live API can change as new launches and metadata are added. This raw collection exercise is separate from the fixed historical IBM course datasets used in downstream labs; do not silently mix these sources without documenting a join key and date/version.